# Scraping Websites, Topic Modeling, XML Edition

## Python Web Scraping — Core Imports

| Library | Type | Role |
|---|---|---|
| `requests` | 3rd party | Fetches raw HTML from the web |
| `BeautifulSoup` (bs4) | 3rd party | Parses HTML so you can query it |
| `urljoin` (urllib.parse) | built-in | Resolves relative URLs to absolute |
| `time` | built-in | `time.sleep()` adds polite delay between requests |

`requests` gets the HTML; `BeautifulSoup` reads it. Neither knows how to do the other's job.

In [3]:
# the requests library is used to fetch the HTML content of the webpage
import requests
# BeautifulSoup is used to parse the HTML and extract the links
from bs4 import BeautifulSoup
from urllib.parse import urljoin
# built in library: courtesy delay between HTTP requests 
# so you don't hammer the server with dozens of rapid-fire requests
import time

In [4]:
# declare the base URL as a variable
BASE = "https://www.history.pitt.edu"

In [5]:
# Step 1: collect article URLs
links = []
for page in range(2):
    soup = BeautifulSoup(requests.get(f"{BASE}/news?page={page}").content, "lxml")
    links += [urljoin(BASE, a["href"]) for a in soup.select("h3 a") if a["href"].startswith("/")]

# Step 2: scrape each article into a list of dicts
articles = []
for url in links[:30]:
    soup = BeautifulSoup(requests.get(url).content, "lxml")
    articles.append({
        "title": soup.find("h1").get_text(strip=True),
        "body":  " ".join(p.get_text(strip=True) for p in soup.find_all("p")),
        "url":   url
    })
    time.sleep(1)

In [8]:
from collections import Counter
import re

all_text = " ".join(a["body"] for a in articles).lower()
words = re.findall(r'\b[a-z]{4,}\b', all_text)   # words 4+ chars

stopwords = {"that","this","with","from","have","been","their",
             "they","will","also","which","were","about","pittsburgh",
             "university","history","department"}

counts = Counter(w for w in words if w not in stopwords)
counts.most_common(20)

[('school', 61),
 ('dietrich', 56),
 ('arts', 55),
 ('pittsburghkenneth', 50),
 ('sciencesdepartment', 50),
 ('wesley', 50),
 ('posvar', 50),
 ('hallpittsburgh', 50),
 ('american', 34),
 ('pitt', 30),
 ('award', 28),
 ('visitapply', 25),
 ('ipsum', 25),
 ('goode', 24),
 ('professor', 22),
 ('latin', 22),
 ('black', 22),
 ('studies', 19),
 ('work', 19),
 ('benjamin', 16)]